In [1]:
import pandas as pd
import numpy as np

In [2]:
car = pd.read_csv("Car_Prediction_Dataset_Project.csv")

In [3]:
car.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [4]:
car.shape

(892, 6)

In [5]:
car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        892 non-null    object
 1   company     892 non-null    object
 2   year        892 non-null    object
 3   Price       892 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: object(6)
memory usage: 41.9+ KB


In [6]:
car["year"].unique()

array(['2007', '2006', '2018', '2014', '2015', '2012', '2013', '2016',
       '2010', '2017', '2008', '2011', '2019', '2009', '2005', '2000',
       '...', '150k', 'TOUR', '2003', 'r 15', '2004', 'Zest', '/-Rs',
       'sale', '1995', 'ara)', '2002', 'SELL', '2001', 'tion', 'odel',
       '2 bs', 'arry', 'Eon', 'o...', 'ture', 'emi', 'car', 'able', 'no.',
       'd...', 'SALE', 'digo', 'sell', 'd Ex', 'n...', 'e...', 'D...',
       ', Ac', 'go .', 'k...', 'o c4', 'zire', 'cent', 'Sumo', 'cab',
       't xe', 'EV2', 'r...', 'zest'], dtype=object)

## Quallity
- year has many non years values
- year object to int
- price value has different value other number
- price object to int
- kms_driven has kms with integers
- kms_driven object to int
- kms_driven has nan values
- fuek_type has nan values
- keep first 3 words of name

# Cleaning

In [7]:
backup = car.copy()

In [8]:
car = car[car["year"].str.isnumeric()]

In [9]:
car["year"] = car["year"].astype(int)

In [10]:
car = car[car["Price"] != "Ask For Price"]

In [11]:
car["Price"] = car["Price"].str.replace(",","").astype(int)

In [12]:
car["kms_driven"] = car["kms_driven"].str.split(" ").str.get(0).str.replace(",","")

In [13]:
car = car[car["kms_driven"].str.isnumeric()]

In [14]:
car["kms_driven"] = car["kms_driven"].astype(int)

In [15]:
car = car[~car["fuel_type"].isna()]

In [16]:
car["name"] = car["name"].str.split(" ").str.slice(0,3).str.join(" ")

In [17]:
car = car.reset_index(drop=True)

In [18]:
car = car[car["Price"] < 6e6].reset_index(drop=True)

In [19]:
car.to_csv("Cleaned Car.csv")

## Model

In [20]:
x = car.drop(columns = "Price")
y = car["Price"]

In [21]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [23]:
ohe = OneHotEncoder()
ohe.fit(x[["name", "company", "fuel_type"]])

OneHotEncoder()

In [24]:
column_trans = make_column_transformer((OneHotEncoder(categories=ohe.categories_),["name", "company", "fuel_type"]), remainder="passthrough")

In [25]:
lr = LinearRegression()

In [26]:
pipe = make_pipeline(column_trans, lr)

In [27]:
pipe.fit(x_train, y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(categories=[array(['Audi A3 Cabriolet', 'Audi A4 1.8', 'Audi A4 2.0', 'Audi A6 2.0',
       'Audi A8', 'Audi Q3 2.0', 'Audi Q5 2.0', 'Audi Q7', 'BMW 3 Series',
       'BMW 5 Series', 'BMW 7 Series', 'BMW X1', 'BMW X1 sDrive20d',
       'BMW X1 xDrive20d', 'Chevrolet Beat', 'Chevrolet Beat...
                                                                            array(['Audi', 'BMW', 'Chevrolet', 'Datsun', 'Fiat', 'Force', 'Ford',
       'Hindustan', 'Honda', 'Hyundai', 'Jaguar', 'Jeep', 'Land',
       'Mahindra', 'Maruti', 'Mercedes', 'Mini', 'Mitsubishi', 'Nissan',
       'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo'],
      dtype=object),
                                                                            array(['Diesel', 'LPG', 'Petrol'], dtype=object)]),
                                                  ['name', 'company',
                                                   'fuel_type'])])),
                ('linearregression', LinearRegression())])

In [28]:
y_pred = pipe.predict(x_test)

In [29]:
r2_score(y_test, y_pred)

0.6964313711001838

In [30]:
scores = []

for i in range(1000):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=i)
    
    lr = LinearRegression()
    pipe = make_pipeline(column_trans, lr)
    
    pipe.fit(x_train, y_train)
    
    y_pred = pipe.predict(x_test)
    
    scores.append(r2_score(y_test, y_pred))

In [31]:
np.argmax(scores)

np.int64(433)

In [32]:
scores[np.argmax(scores)]

0.8456642667912282

In [33]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=np.argmax(scores))
    
lr = LinearRegression()
pipe = make_pipeline(column_trans, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
scores.append(r2_score(y_test, y_pred))
r2_score(y_test, y_pred)

0.8456642667912282

In [34]:
import pickle

In [35]:
pickle.dump(pipe,open("LinearRegressionModel.pkl","wb"))

In [36]:
pipe.predict(pd.DataFrame([["Maruti Suzuki Swift","Maruti",2019,100,"Petrol"]], columns = ["name","company","year","kms_driven","fuel_type"]))

array([459062.10569844])

In [1]:
import os
os.getcwd()

'/Users/shobhitjaiswal/Car Prediction Project'